In [0]:
from pyspark.sql.functions import to_date, try_to_date, split, col, element_at, lit, when, length, concat

In [0]:
silver_bt = spark.table("workspace.default.bronze_bank_transactions")

In [0]:
silver_bt.printSchema()

In [0]:
silver_bt.filter(col("CustomerDOB").isNull()).count()

In [0]:
from pyspark.sql import functions as F

In [0]:
# bad_balance = silver_bt.filter(
#     F.col("CustAccountBalance").isNotNull() &
#     F.expr("try_cast(CustAccountBalance AS DOUBLE)").isNull()
# )

# bad_balance.count()

Checking if <br>

    -CustAccountBalance
    -TransactionAmountINR


are null in general or after typecast becomes null as they are 'a23' and gets casted to null
We first ran F.expr - which basically runs SQL code and returns back pyspark col object
then try_cast for a safe type cast conversion which returns null instead of throwing an error or runtime exception if value cannot be converted.

In [0]:
bad_balance = silver_bt.filter(F.expr("try_cast(CustAccountBalance AS DOUBLE)").isNull())
bad_balance.count()

In [0]:
silver_bt.filter(F.col("CustAccountBalance").isNull()).count()

In [0]:
silver_bt.filter(F.col("TransactionAmountINR").isNull()).count()

In [0]:
silver_bt.filter(F.expr("try_cast(TransactionAmountINR AS DOUBLE)").isNull()).count()

### type casting

In [0]:
# silver_bt = silver_bt.withColumn("CustAccountBalance", F.expr("try_cast(CustAccountBalance AS DOUBLE)"))
silver_bt = silver_bt.withColumn("CustAccuntBalance", F.col("CustAccountBalance").cast("double"))

In [0]:
silver_bt.filter(F.expr("try_cast(CustomerDOB AS DATE)").isNull()).count()

In [0]:
silver_bt.count()

In [0]:
silver_bt.filter(F.col("CustomerDOB").isNull()).count()

In [0]:
silver_bt.select("CustomerDOB").show(10)

In [0]:
silver_bt.withColumn("DOB_parse", try_to_date(F.col("CustomerDOB"), "d/M/yy")).select("CustomerDOB", "DOB_parse").show(10)

In [0]:
temp_df = silver_bt.withColumn("dob_parts", split(col("CustomerDOB"), "/"))
temp_df.select("CustomerDOB", "dob_parts").show(10)

In [0]:
# temp_df = temp_df.select("dob_parts").element_at("dob_parts",3).length()
# temp_df = temp_df.concat(temp_df, "19")

# temp_df = temp_df.withColumn("dob_year_raw", element_at(col("dob_parts"), 3))
# temp_df.select("CustomerDOB", "dob_parts", "dob_year_raw").show(10)

In [0]:
# from pyspark.sql.functions import size

# temp_df.filter(size(col("dob_parts")) != 3).select("CustomerDOB").show(20, truncate=False)


In [0]:
silver_bt.filter(col("CustomerDOB") == 'nan').count()

converted nan value to null

In [0]:
silver_bt = silver_bt.withColumn(
    "CustomerDOB", 
    when(col("CustomerDOB")=='nan', lit(None)).otherwise(col("CustomerDOB")))

In [0]:
silver_bt.filter(col("CustomerDOB").isNull()).count()

In [0]:
temp_df = silver_bt.withColumn("dob_parts", split(col("CustomerDOB"), "/"))

In [0]:
temp_df.select("CustomerDOB", "dob_parts").show(5)

In [0]:
# temp_df.filter(col('CustomerDOB').isNull()).select("CustomerDOB", "dob_parts").show(5)

In [0]:
# # Some checks for Null in customerDOB
# temp_df.count()
# silver_bt.count()
# temp_df.filter(col("CustomerDOB").isNull()).count()
# temp_df.filter(size(col("dob_parts")) != 3).count()
# temp_df.filter(col("dob_parts").isNull()).select(size(col("dob_parts"))).show(5)
# temp_df.filter(col("dob_parts").isNull() | (size(col("dob_parts")) != 3)).count()

In [0]:
temp_df = temp_df.withColumn("dob_raw_year", element_at(col("dob_parts"),3))

In [0]:
temp_df.select("CustomerDOB","dob_parts", "dob_raw_year").show(5)

which rows have a 2-digit year (need the "19" prefix) versus the rare 4-digit ones like the 1/1/1800 row

In [0]:
temp_df = temp_df.withColumn("dob_year_lebgth", length(col("dob_raw_year")))

In [0]:
temp_df.select("dob_year_lebgth").distinct().show()

In [0]:
temp_df = temp_df.withColumn("dob_year_corrected", when(col("dob_year_lebgth")==2, concat(lit("19"),col("dob_raw_year"))).otherwise(col("dob_raw_year")))

In [0]:
temp_df.select("CustomerDOB", "dob_raw_year", "dob_year_lebgth", "dob_year_corrected").show(10)

In [0]:
temp_df.filter(col("dob_year_lebgth") == 4).select("CustomerDOB", "dob_raw_year", "dob_year_lebgth", "dob_year_corrected").show(10)

if using element_at indexing starts from 1, but if usinng [] array indexing it starts from 0

In [0]:
temp_df = temp_df.withColumn("CustomerDOB_corrected", concat(col("dob_parts")[0], lit("/"), col("dob_parts")[1], lit("/"), col("dob_year_corrected")))

In [0]:
temp_df.select("CustomerDOB", "CustomerDOB_corrected").show(10)

formatting customerDOB_corrected

In [0]:
temp_df = temp_df.withColumn("CustomerDOB_final", try_to_date(col("CustomerDOB_corrected"), "d/M/yyyy"))

In [0]:
temp_df.select("CustomerDOB", "CustomerDOB_corrected", "CustomerDOB_final").show(10)

In [0]:
# quality check for null
temp_df.filter(col("CustomerDOB_corrected").isNotNull() & col("CustomerDOB_final").isNull()).count()